In [1]:
from dotenv import load_dotenv

print("dotenv imported successfully")

dotenv imported successfully


In [2]:
import os
import sys
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv

# ------------------------------------------------------------
# Load environment variables from the .env file
# ------------------------------------------------------------
# The .env file should contain:
# BC_ODATA_URL=your_business_central_odata_url
# BC_USERNAME=your_username
# BC_PASSWORD_OR_ACCESS_KEY=your_password_or_access_key
# OUTPUT_CSV=optional_output_file_name.csv
# VERIFY_SSL=true
load_dotenv(override=True)

# ------------------------------------------------------------
# Read configuration values from environment variables
# ------------------------------------------------------------
BC_ODATA_URL = os.getenv("BC_ODATA_URL")
BC_ODATA_URL_2 = os.getenv("BC_ODATA_URL_2")
BC_USERNAME = os.getenv("BC_USERNAME")
BC_PASSWORD_OR_ACCESS_KEY = os.getenv("BC_PASSWORD_OR_ACCESS_KEY")

# Default CSV file name if OUTPUT_CSV is not provided in .env
OUTPUT_CSV = os.getenv("OUTPUT_CSV", "business_central_export.csv")

# Convert VERIFY_SSL value from string to boolean
# true  -> SSL certificate will be verified
# false -> SSL certificate verification will be skipped
VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() == "true"


def validate_env():
    """
    Validates that all required environment variables are available.

    If any required variable is missing, the script stops immediately.
    This prevents the script from running with incomplete configuration.
    """

    print("[STEP 1] Validating environment variables...")

    required_vars = {
        "BC_ODATA_URL": BC_ODATA_URL,
        "BC_ODATA_URL_2": BC_ODATA_URL_2,
        "BC_USERNAME": BC_USERNAME,
        "BC_PASSWORD_OR_ACCESS_KEY": BC_PASSWORD_OR_ACCESS_KEY,
    }

    missing = [key for key, value in required_vars.items() if not value]

    if missing:
        print("\n[ERROR] Missing required environment variables:")
        for key in missing:
            print(f"  - {key}")

        print("\nPlease check your .env file and try again.")
        sys.exit(1)

    print("[OK] All required environment variables are present.")


def get_output_file_path(file_name):
    """
    Returns the full CSV file path inside the current working directory.

    Example:
    If current working directory is:
      D:/Projects/BCExport

    and file_name is:
      export.csv

    then this function returns:
      D:/Projects/BCExport/export.csv
    """

    print("\n[STEP 2] Preparing output file location...")
    current_working_directory = os.getcwd()

    # Keep only the file name, even if user accidentally gives a path.
    # This ensures the file is saved in the current working directory.
    safe_file_name = os.path.basename(file_name)
    output_path = os.path.join(current_working_directory, safe_file_name)

    print(f"[INFO] Current working directory: {current_working_directory}")
    print(f"[INFO] Output CSV file name: {safe_file_name}")
    print(f"[INFO] Full output path: {output_path}")

    return output_path


def create_session():
    """
    Creates and configures a requests session for Business Central.

    The session contains:
    - Basic authentication
    - JSON response header
    - Read-only intent header

    Important:
    This script only uses GET requests.
    It does not use POST, PATCH, PUT, or DELETE.
    So it does not create, update, or delete Business Central records.
    """

    print("\n[STEP 3] Creating secure HTTP session...")
    session = requests.Session()

    # Add Basic Authentication credentials.
    # Depending on your Business Central setup, this may be:
    # - username + password
    # - username + web service access key
    session.auth = HTTPBasicAuth(BC_USERNAME, BC_PASSWORD_OR_ACCESS_KEY)

    # Accept JSON response from Business Central.
    # Data-Access-Intent: ReadOnly tells Business Central that
    # this request is intended only for reading data.
    session.headers.update(
        {
            "Accept": "application/json",
            "Data-Access-Intent": "ReadOnly",
        }
    )

    print("[OK] HTTP session created.")
    print("[INFO] Request method used by this script: GET only")
    print("[INFO] Data access intent: ReadOnly")

    return session


def fetch_all_records(url):
    """
    Fetches all records from the Business Central OData endpoint.

    Business Central may return data in multiple pages.
    If more pages are available, the response contains '@odata.nextLink'.

    This function keeps fetching records until there is no next page left.
    """

    print("\n[STEP 4] Starting data fetch from Business Central...")
    print(f"[INFO] Source OData URL: {url}")
    print(f"[INFO] SSL verification enabled: {VERIFY_SSL}")

    all_records = []
    next_url = url
    page_number = 1

    session = create_session()

    while next_url:
        print(f"\n[FETCH] Reading page {page_number}...")
        print(f"[FETCH] URL: {next_url}")

        try:
            # This is the actual read-only request.
            # GET means fetch/read data only.
            response = session.get(
                next_url,
                timeout=60,
                verify=VERIFY_SSL,
            )

        except requests.exceptions.SSLError as error:
            print("\n[ERROR] SSL certificate verification failed.")
            print("Possible solutions:")
            print("  1. Use a valid SSL certificate on the server.")
            print("  2. For local/testing only, set VERIFY_SSL=false in .env.")
            print(f"\nTechnical details: {error}")
            sys.exit(1)

        except requests.exceptions.ConnectionError as error:
            print("\n[ERROR] Could not connect to the Business Central server.")
            print("Please check:")
            print("  1. Server URL")
            print("  2. Port number")
            print("  3. Network/VPN connectivity")
            print("  4. Firewall settings")
            print(f"\nTechnical details: {error}")
            sys.exit(1)

        except requests.exceptions.Timeout as error:
            print("\n[ERROR] Request timed out.")
            print("The server took too long to respond.")
            print("Try again later or export a smaller filtered dataset.")
            print(f"\nTechnical details: {error}")
            sys.exit(1)

        except requests.exceptions.RequestException as error:
            print("\n[ERROR] Unexpected request error occurred.")
            print(f"Technical details: {error}")
            sys.exit(1)

        print(f"[INFO] HTTP status code: {response.status_code}")

        if response.status_code != 200:
            print("\n[ERROR] Request failed.")

            if response.status_code == 401:
                print(
                    "[CAUSE] Unauthorized. Username/password/access key may be incorrect."
                )
            elif response.status_code == 403:
                print(
                    "[CAUSE] Forbidden. Your user may not have permission to access this OData service."
                )
            elif response.status_code == 404:
                print(
                    "[CAUSE] Not found. The OData URL, company, or service name may be incorrect."
                )
            elif response.status_code >= 500:
                print("[CAUSE] Server-side error from Business Central.")

            print(f"\nResponse from server:\n{response.text}")
            response.raise_for_status()

        try:
            data = response.json()

        except ValueError:
            print("\n[ERROR] Server response is not valid JSON.")
            print(
                "Please check if the URL is a valid Business Central OData JSON endpoint."
            )
            print(f"\nResponse text:\n{response.text}")
            sys.exit(1)

        # In OData responses, actual records are usually inside the "value" key.
        records = data.get("value", [])
        print(f"[OK] Records fetched from page {page_number}: {len(records)}")
        all_records.extend(records)
        print(f"[INFO] Total records fetched so far: {len(all_records)}")

        # If Business Central has more pages, it provides @odata.nextLink.
        next_url = data.get("@odata.nextLink")
        if next_url:
            print("[INFO] More records available. Moving to next page...")
        else:
            print("[INFO] No more pages found.")

        page_number += 1

    print("\n[OK] Data fetch completed.")
    print(f"[INFO] Total records fetched: {len(all_records)}")
    return all_records


def save_to_csv(records, output_path):
    """
    Converts fetched Business Central records into a CSV file.

    Steps:
    1. Convert JSON records into a pandas DataFrame.
    2. Remove technical OData metadata columns.
    3. Save the DataFrame as a CSV file.
    """

    print("\n[STEP 5] Preparing data for CSV export...")
    if not records:
        print("[WARNING] No records found.")
        print("[INFO] CSV file was not created because there is no data to export.")
        return

    print("[INFO] Converting JSON records into table format...")

    # Converts nested JSON into a flat table where possible.
    df = pd.json_normalize(records)
    print(f"[OK] Data converted successfully.")
    print(f"[INFO] Number of rows: {len(df)}")
    print(f"[INFO] Number of columns before cleanup: {len(df.columns)}")

    # Remove technical OData metadata columns such as @odata.etag.
    metadata_columns = [col for col in df.columns if col.startswith("@odata.")]

    if metadata_columns:
        print(f"[INFO] Removing OData metadata columns: {metadata_columns}")
        df.drop(columns=metadata_columns, inplace=True)
    else:
        print("[INFO] No OData metadata columns found.")

    print(f"[INFO] Number of columns after cleanup: {len(df.columns)}")
    print("\n[STEP 6] Saving data to CSV file...")

    try:
        # utf-8-sig works well with Microsoft Excel.
        df.to_csv(output_path, index=False, encoding="utf-8-sig")

    except PermissionError:
        print("\n[ERROR] Permission denied while saving CSV file.")
        print("Please check:")
        print("  1. The CSV file is not already open in Excel.")
        print("  2. You have write permission in the current folder.")
        print(f"  3. Output path: {output_path}")
        sys.exit(1)

    except Exception as error:
        print("\n[ERROR] Failed to save CSV file.")
        print(f"Technical details: {error}")
        sys.exit(1)

    print("\n[SUCCESS] Export completed successfully.")
    print(f"[SUCCESS] Rows exported: {len(df)}")
    print(f"[SUCCESS] Columns exported: {len(df.columns)}")
    print(f"[SUCCESS] CSV saved at: {output_path}")




def main():

    validate_env()
    output_path = get_output_file_path(OUTPUT_CSV)

    # -----------------------------
    # FETCH DATA
    # -----------------------------
    records1 = fetch_all_records(BC_ODATA_URL)
    records2 = fetch_all_records(BC_ODATA_URL_2)

    df1 = pd.json_normalize(records1)
    df2 = pd.json_normalize(records2)

    print("DF1 columns:", df1.columns.tolist())
    print("DF2 columns:", df2.columns.tolist())

    # -----------------------------
    # CLEAN DF1 (Invoice Line)
    # -----------------------------
    selected_cols_df1 = [
        'Amount',
        'AmountIncludingVAT',
        'Description',
        'DocumentNo',
        'BilltoCustomerNo',
        'Quantity',
        'PostingDate',
        'LocationCode',
        'GSTAssessableValueLCY',
        'InvDiscountAmount',
        'ItemCategoryCode'
    ]

    df1_clean = df1[[col for col in selected_cols_df1 if col in df1.columns]]

    # -----------------------------
    # CLEAN DF2 (Invoice Header)
    # IMPORTANT: No column may NOT exist, so we handle safely
    # -----------------------------

    # Find correct key column automatically
    possible_keys = ['No', 'No_', 'no', 'DocumentNo', 'Document_No']

    header_key = None
    for col in possible_keys:
        if col in df2.columns:
            header_key = col
            break

    if header_key is None:
        print("\n[ERROR] No valid key column found in df2!")
        print("Available columns:", df2.columns.tolist())
        sys.exit(1)

    print(f"\n[INFO] Using header key column: {header_key}")

    selected_cols_df2 = [
        header_key,
        'billToName',
        'billToCity', 
         'state',
    ]

    df2_clean = df2[[col for col in selected_cols_df2 if col in df2.columns]]

    # -----------------------------
    # SAFE MERGE
    # -----------------------------
    df_merged = df1_clean.merge(
        df2_clean,
        left_on='DocumentNo',
        right_on=header_key,
        how='left'
    )

    # -----------------------------
    # OUTPUT FILES
    # -----------------------------
    df1_clean.to_csv("InvoiceLine.csv", index=False, encoding="utf-8-sig")
    df2_clean.to_csv("InvoiceHeader.csv", index=False, encoding="utf-8-sig")
    df_merged.to_csv("InvoiceMerged.csv", index=False, encoding="utf-8-sig")

    print("\n[SUCCESS] Files created successfully!")
    print("InvoiceLine.csv")
    print("InvoiceHeader.csv")
    print("InvoiceMerged.csv")
    # print(df2_clean)

    print("\nMerged Shape:", df_merged.shape)

    return df1_clean, df2_clean, df_merged


if __name__ == "__main__":
    
    main  ()
    

[STEP 1] Validating environment variables...
[OK] All required environment variables are present.

[STEP 2] Preparing output file location...
[INFO] Current working directory: c:\Users\naina\Downloads\EDA report of sales
[INFO] Output CSV file name: dataset.csv
[INFO] Full output path: c:\Users\naina\Downloads\EDA report of sales\dataset.csv

[STEP 4] Starting data fetch from Business Central...
[INFO] Source OData URL: https://oswalsolarerp.in:7048/BC240/ODataV4/Company(%27Oswal%20Solar%20Structure%20Pvt.%20Ltd%27)/SalesInvoiceLine
[INFO] SSL verification enabled: True

[STEP 3] Creating secure HTTP session...
[OK] HTTP session created.
[INFO] Request method used by this script: GET only
[INFO] Data access intent: ReadOnly

[FETCH] Reading page 1...
[FETCH] URL: https://oswalsolarerp.in:7048/BC240/ODataV4/Company(%27Oswal%20Solar%20Structure%20Pvt.%20Ltd%27)/SalesInvoiceLine
[INFO] HTTP status code: 200
[OK] Records fetched from page 1: 15462
[INFO] Total records fetched so far: 15462

In [3]:
df1_clean, df2_clean, df_merged = main()

[STEP 1] Validating environment variables...
[OK] All required environment variables are present.

[STEP 2] Preparing output file location...
[INFO] Current working directory: c:\Users\naina\Downloads\EDA report of sales
[INFO] Output CSV file name: dataset.csv
[INFO] Full output path: c:\Users\naina\Downloads\EDA report of sales\dataset.csv

[STEP 4] Starting data fetch from Business Central...
[INFO] Source OData URL: https://oswalsolarerp.in:7048/BC240/ODataV4/Company(%27Oswal%20Solar%20Structure%20Pvt.%20Ltd%27)/SalesInvoiceLine
[INFO] SSL verification enabled: True

[STEP 3] Creating secure HTTP session...
[OK] HTTP session created.
[INFO] Request method used by this script: GET only
[INFO] Data access intent: ReadOnly

[FETCH] Reading page 1...
[FETCH] URL: https://oswalsolarerp.in:7048/BC240/ODataV4/Company(%27Oswal%20Solar%20Structure%20Pvt.%20Ltd%27)/SalesInvoiceLine
[INFO] HTTP status code: 200
[OK] Records fetched from page 1: 15462
[INFO] Total records fetched so far: 15462

In [4]:
print(df1_clean.shape)
print(df2_clean.shape)
print(df_merged.shape)

(15462, 11)
(6206, 4)
(15462, 15)


In [5]:
type(main)

function

In [6]:
df1_clean, df2_clean, df_merged = main()

[STEP 1] Validating environment variables...
[OK] All required environment variables are present.

[STEP 2] Preparing output file location...
[INFO] Current working directory: c:\Users\naina\Downloads\EDA report of sales
[INFO] Output CSV file name: dataset.csv
[INFO] Full output path: c:\Users\naina\Downloads\EDA report of sales\dataset.csv

[STEP 4] Starting data fetch from Business Central...
[INFO] Source OData URL: https://oswalsolarerp.in:7048/BC240/ODataV4/Company(%27Oswal%20Solar%20Structure%20Pvt.%20Ltd%27)/SalesInvoiceLine
[INFO] SSL verification enabled: True

[STEP 3] Creating secure HTTP session...
[OK] HTTP session created.
[INFO] Request method used by this script: GET only
[INFO] Data access intent: ReadOnly

[FETCH] Reading page 1...
[FETCH] URL: https://oswalsolarerp.in:7048/BC240/ODataV4/Company(%27Oswal%20Solar%20Structure%20Pvt.%20Ltd%27)/SalesInvoiceLine
[INFO] HTTP status code: 200
[OK] Records fetched from page 1: 15462
[INFO] Total records fetched so far: 15462

In [7]:
print(df1_clean.columns.tolist())

['Amount', 'AmountIncludingVAT', 'Description', 'DocumentNo', 'BilltoCustomerNo', 'Quantity', 'PostingDate', 'LocationCode', 'GSTAssessableValueLCY', 'InvDiscountAmount', 'ItemCategoryCode']


In [8]:
df1_clean.sort_values(by='Amount', ascending=False)

,Amount,AmountIncludingVAT,Description,DocumentNo,BilltoCustomerNo,Quantity,PostingDate,LocationCode,GSTAssessableValueLCY,InvDiscountAmount,ItemCategoryCode
9704,15950000.0,15950000.0,SUPPLY OF SOLAR ROOF TOP SOLUTION 1000 KWP,SINV25-03815,C000460,1.0,2026-01-02,FG,0,0,EPC PROJECT
14493,15650460.0,15650460.0,SOLAR PANEL 520 WP MONO HALF CUT DCR BIFACIAL,SIOS-2627-00533,C000592,1200.0,2026-05-13,IN-SITE,0,0,FG-SOLAR PANEL TB
14326,15650460.0,15650460.0,SOLAR PANEL 520 WP MONO HALF CUT DCR BIFACIAL,SIOS-2627-00477,C000592,1200.0,2026-05-07,PROJECT,0,0,FG-SOLAR PANEL TB
14752,13042050.0,13042050.0,SOLAR PANEL 520 WP MONO HALF CUT DCR BIFACIAL,SIOS-2627-00627,C000592,1000.0,2026-05-22,IN-SITE,0,0,FG-SOLAR PANEL TB
14844,13042050.0,13042050.0,SOLAR PANEL 520 WP MONO HALF CUT DCR BIFACIAL,SIOS-2627-00662,C000592,1000.0,2026-05-25,IN-SITE,0,0,FG-SOLAR PANEL TB
...,...,...,...,...,...,...,...,...,...,...,...
10,0.0,0.0,Shipment No. DS24-01747:,SINV24-02300,,0.0,2025-02-03,FG,0,0,
12,0.0,0.0,Shipment No. DS24-01748:,SINV24-02301,,0.0,2025-02-03,,0,0,
14,0.0,0.0,Shipment No. DS24-01749:,SINV24-02302,,0.0,2025-02-01,FG,0,0,
14352,0.0,0.0,Shipment No. DS26-00433:,SIOS-2627-00482,,0.0,2026-05-07,WH-KUTAIL,0,0,


TOP PRODUCTS WITH SALES AMOUNT

In [9]:
import plotly.graph_objects as go
import pandas as pd

# =====================================================
# TOP 10 PRODUCT CATEGORIES
# =====================================================

top_products = (
    df_merged[
        df_merged['ItemCategoryCode'].fillna('').str.strip() != ''
    ]
    .groupby('ItemCategoryCode', as_index=False)['Amount']
    .sum()
    .sort_values('Amount', ascending=False)
    .head(10)
)

df = top_products.copy()

# =====================================================
# FORMATTING FUNCTIONS
# =====================================================

def to_crore(x):
    return f"₹ {x/1e7:.2f} Cr"

def indian_format(x):
    x = int(round(x))
    s = str(x)

    if len(s) <= 3:
        return "₹ " + s

    last3 = s[-3:]
    rest = s[:-3]

    parts = []

    while len(rest) > 2:
        parts.insert(0, rest[-2:])
        rest = rest[:-2]

    if rest:
        parts.insert(0, rest)

    return "₹ " + ",".join(parts) + "," + last3

# =====================================================
# PREPARE DATA
# =====================================================

df.insert(0, 'Rank', range(1, len(df) + 1))

df['Amount_Cr'] = df['Amount'].apply(to_crore)
df['Rounded_Amount'] = df['Amount'].apply(indian_format)

# =====================================================
# TOTAL ROW
# =====================================================

total_amount = df['Amount'].sum()

total_row = pd.DataFrame({
    'Rank': [''],
    'ItemCategoryCode': ['TOTAL'],
    'Amount': [total_amount],
    'Amount_Cr': [to_crore(total_amount)],
    'Rounded_Amount': [indian_format(total_amount)]
})

# =====================================================
# FINAL DATAFRAME
# =====================================================

final_df = pd.concat([df, total_row], ignore_index=True)

# =====================================================
# ROW COLORS
# =====================================================

row_colors = ["#DDF5E3"] * len(final_df)
row_colors[-1] = "#008037"

# =====================================================
# BOLD TOTAL ROW
# =====================================================

rank_col = list(final_df['Rank'])
item_col = list(final_df['ItemCategoryCode'])
amount_cr_col = list(final_df['Amount_Cr'])
rounded_col = list(final_df['Rounded_Amount'])

rank_col[-1] = "<b></b>"
item_col[-1] = "<b>TOTAL</b>"
amount_cr_col[-1] = f"<b>{amount_cr_col[-1]}</b>"
rounded_col[-1] = f"<b>{rounded_col[-1]}</b>"

# =====================================================
# TABLE
# =====================================================

fig = go.Figure(data=[go.Table(

    columnwidth=[80, 280, 180, 220],

    header=dict(
        values=[
            '<b>Rank</b>',
            '<b>Item Category</b>',
            '<b>Amount</b>',
            '<b>Full Rounded Amount</b>'
        ],

        fill_color="#008037",

        font=dict(
            color='white',
            size=14
        ),

        align='center',
        height=40
    ),

    cells=dict(

        values=[
            rank_col,
            item_col,
            amount_cr_col,
            rounded_col
        ],

        fill_color=[
            row_colors,
            row_colors,
            row_colors,
            row_colors
        ],

        align='center',

        font=dict(
            color=[
                ['black']*(len(final_df)-1)+['white'],
                ['black']*(len(final_df)-1)+['white'],
                ['black']*(len(final_df)-1)+['white'],
                ['black']*(len(final_df)-1)+['white']
            ],
            size=12
        ),

        height=35
    )
)])

# =====================================================
# LAYOUT
# =====================================================

fig.update_layout(

    title={
        'text': '<b>Top 10 Product Categories by Sales Amount</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(
            size=22,
            color='#0E0E0E'
        )
    },

    paper_bgcolor='white',
    plot_bgcolor='white',

    width=1000,
    height=50 + (len(final_df) * 35),

    margin=dict(
        l=20,
        r=20,
        t=80,
        b=20
    )
)

fig.show()

TOP 10 CUSTOMERS BY REVENUE 

In [84]:
print(df1_clean.columns)

Index(['Amount', 'AmountIncludingVAT', 'Description', 'DocumentNo',
       'BilltoCustomerNo', 'Quantity', 'PostingDate', 'LocationCode',
       'GSTAssessableValueLCY', 'InvDiscountAmount', 'ItemCategoryCode'],
      dtype='str')


In [85]:
df['Customer_Label'] = (
    df_merged['BilltoCustomerNo'].astype(str)
    + ' - '
    + df_merged['billToName'].astype(str)
)

top_customers = (
    df.groupby('Customer_Label', as_index=False)['Amount']
    .sum()
    .sort_values('Amount', ascending=False)
    .head(10)
)

In [12]:
df_merged[df_merged['BilltoCustomerNo'] == 'C000106'][['BilltoCustomerNo','billToName']].drop_duplicates()

,BilltoCustomerNo,billToName
7,C000106,OSWAL PUMPS LIMITED (SOLAR PV)
3892,C000106,OSWAL PUMPS LIMITED


In [86]:
top_customers = (
    df_merged.groupby(
        ['BilltoCustomerNo', 'billToName'],
        as_index=False
    )['Amount']
    .sum()
    .sort_values('Amount', ascending=False)
    .head(10)
)

In [14]:
top_customers['Customer_Label'] = (
    top_customers['billToName']
    + ' ('
    + top_customers['BilltoCustomerNo']
    + ')'
)

TOP CUSTOMERS BY REVENUE

In [87]:
import plotly.express as px
import pandas as pd

df = df_merged.copy()

# ---------------- CLEAN DATA ----------------
df = df.dropna(subset=['BilltoCustomerNo', 'Amount'])
df = df[df['BilltoCustomerNo'].astype(str).str.strip() != '0']

df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df = df.dropna(subset=['Amount'])

# ---------------- TOP 10 CUSTOMERS ----------------
top_customers = (
    df.groupby(['BilltoCustomerNo', 'billToName'], as_index=False)['Amount']
    .sum()
    .sort_values('Amount', ascending=False)
    .head(10)
)

# Convert to Crores
top_customers['Revenue_Cr'] = top_customers['Amount'] / 1e7
max_rev = top_customers['Revenue_Cr'].max()

# ---------------- PLOT ----------------
fig = px.bar(
    top_customers,
    x='Revenue_Cr',
    y='billToName',
    orientation='h',
    text='Revenue_Cr',
    title='👤 Top 10 Customers by Revenue',
    custom_data=['billToName','BilltoCustomerNo']   # ✅ IMPORTANT
)

fig.update_traces(
     texttemplate='₹ %{text:.2f} Cr',
    textposition='outside',
    cliponaxis=False,
    marker=dict(color='#007A33'),
        textfont=dict(
        size=16,
        color="black"
    ),
    hovertemplate=
        '<b>Name:</b> %{customdata[0]}<br>' +
        '<b>Customer No:</b> %{customdata[1]}<br>'
)
fig.update_layout(

        hoverlabel=dict(
        bgcolor="#FFFDD0",
        font_color="black",
    ),
    height=800,
    width=1500,

    title={
        'text': '👤 Top 10 Customers by Revenue',
        'x': 0.5,
        'xanchor': 'center',
      'font': dict(size=24),
    },

    margin=dict(
        l=150,
        r=250,
        t=100,
        b=80
        ),

    yaxis=dict(

        autorange="reversed",
        title="Bill to Name",
        title_font=dict(size=18, family ="Arial Black"),
        tickfont=dict(size=14, family="Arial Black"),
    ),

    xaxis=dict(
        title='Revenue (₹ Crores)',
        tickformat='.2f',
        ticksuffix=' Cr',
        title_font=dict(size=18, family="Arial Black"),
        tickfont=dict(size=14, family="Arial Black"),
        range=[0, max_rev * 1.25]
    ),
)

fig.show()

In [16]:
df[['billToCity','state']].head()

,billToCity,state
0,ANANTPUR,AP
1,BARAWALA,HR
2,BARAWALA,HR
3,PANIPAT,HR
4,PANIPAT,HR


TOP 10 CITIES BY REVENUE

In [17]:
print(df_merged.columns.tolist())

['Amount', 'AmountIncludingVAT', 'Description', 'DocumentNo', 'BilltoCustomerNo', 'Quantity', 'PostingDate', 'LocationCode', 'GSTAssessableValueLCY', 'InvDiscountAmount', 'ItemCategoryCode', 'no', 'billToName', 'billToCity', 'state']


In [88]:
import plotly.express as px
import pandas as pd

df = df_merged.copy()

# ---------------- CLEAN DATA ----------------
df = df.dropna(subset=['billToCity', 'Amount'])

df['Bill-to City'] = df['billToCity'].astype(str).str.strip()
df = df[df['billToCity'] != '']

df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df = df.dropna(subset=['Amount'])

# ---------------- TOP 10 CITIES ----------------
top_cities = (
    df.groupby(['billToCity', 'state'], as_index=False)['Amount']
      .sum()
      .sort_values('Amount', ascending=False)
      .head(10)
)

top_cities['Revenue_Cr'] = top_cities['Amount'] / 1e7

# ---------------- PLOT ----------------
fig = px.bar(
    top_cities,
    x='Revenue_Cr',
    y='billToCity',
    orientation='h',
    text='Revenue_Cr',
    title=' Top 10 Cities by Revenue',
      custom_data=['state']

)

fig.update_traces(
    texttemplate='₹ %{text:.2f} Cr',
    textposition='outside',
    cliponaxis=False,
    marker=dict(color='#007A33'),
    hovertemplate=
        '<b>State:</b> %{customdata[0]}<br>' 
    
)

max_rev = top_cities['Revenue_Cr'].max()

fig.update_layout(
    hoverlabel=dict(
        bgcolor="#F5F5DC",   # cream color
        font_size=14,
        font_family="Arial",
    ),

    title={
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=24)
    },

    yaxis=dict(

        autorange='reversed',
        title='BilltoCity',
        title_font=dict(size=18,family = "Arial Black"),
        tickfont=dict(size=14,family = "Arial Black"),
    ),

    xaxis=dict(
        title='Revenue (₹ Crores)',
        range=[0, max_rev * 1.4],
        tickprefix='₹ ',
        tickformat='.2f',
        ticksuffix=' Cr',
        title_font=dict(size=18,family = "Arial Black"),
        tickfont=dict(size=14,family = "Arial Black"),

    ),
     height=900,   # 🔥 increase height
    width=1800,

    margin=dict(
        l=180,
        r=350,
        t=80,
        b=80
    )
)

fig.show()  

STATE WISE ANALYSIS

In [ ]:
import plotly.express as px
import pandas as pd

df = df_merged.copy()

# ---------------- CLEAN DATA ----------------
df = df.dropna(subset=['state', 'Amount'])

df['state'] = df['state'].astype(str).str.strip()
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df = df.dropna(subset=['Amount'])

# ---------------- GROUP ----------------
state_sales = (
    df.groupby('state', as_index=False)['Amount']
      .sum()
      .sort_values('Amount', ascending=False)
)

# Convert to Crores
state_sales['Revenue_Cr'] = state_sales['Amount'] / 1e7

# ---------------- PLOT ----------------
fig = px.bar(
    state_sales,
    x='Revenue_Cr',
    y='state',
    orientation='h',
    text='Revenue_Cr',
    title=' State-wise Revenue (In Crores)'
)

# ---------------- STYLE ----------------
fig.update_traces(
    texttemplate='₹ %{text:.2f} Cr',
    textposition='outside',
    marker=dict(color='#007A33'),
      hoverinfo="skip"

)

max_val = state_sales['Revenue_Cr'].max()

fig.update_layout(
    
    # ⭐ TITLE CENTER + BIG
    title=dict(
        x=0.5,
        font=dict(size=26, family="Arial Black")
    ),
    hovermode=False,
    # ⭐ Y AXIS (GOOD FONT)
    yaxis=dict(
        autorange='reversed',
        title=dict(
            text='STATES',
            font=dict(size=20, family="Calibri", color=" Arial black")
        ),
        tickfont=dict(size=14, family="Arial")
    ),

    # ⭐ X AXIS (CRORES FORMATTING)
    xaxis=dict(
        title=dict(
            text='Revenue (₹ Crores)',
            font=dict(size=20, family="Calibri", color=" Arial black")
        ),
        tickfont=dict(size=14),
        tickprefix='₹ ',
        ticksuffix=' Cr',
        range=[0, max_val * 1.3]
    ),
     height=900,   
    width=1600 ,
    margin=dict(l=150, r=50, t=80, b=50)
)

fig.show()

WATT WISE ANALYSIS

In [20]:
df = df_merged.copy()

df['Watt'] = df['Description'].astype(str).str.extract(r'(\d+)\s*W', expand=False)
df['Watt'] = pd.to_numeric(df['Watt'], errors='coerce')

df = df.dropna(subset=['Watt', 'Amount'])

In [21]:
watt_wise = df.groupby('Watt', as_index=False)['Amount'].sum()

In [90]:
import pandas as pd
import plotly.express as px

df = df_merged.copy()

# -------- EXTRACT WATT --------
df['Watt'] = df['Description'].astype(str).str.extract(r'(\d+)\s*W')
df['Watt'] = pd.to_numeric(df['Watt'], errors='coerce')

df = df.dropna(subset=['Watt', 'Amount'])

# -------- GROUP --------
watt_wise = df.groupby('Watt', as_index=False)['Amount'].sum()

# convert to crores
watt_wise['Revenue_Cr'] = watt_wise['Amount'] / 1e7

watt_wise = watt_wise.sort_values('Revenue_Cr', ascending=True)

watt_wise['Watt_label'] = watt_wise['Watt'].astype(int).astype(str) + "W"

# -------- PLOT --------
fig = px.bar(
    watt_wise,
    x='Revenue_Cr',
    y='Watt_label',
    orientation='h',
    text='Revenue_Cr',
    title= 'Watt-wise Analysis',
)

# -------- STYLE UPDATE --------
fig.update_traces(
    texttemplate='₹ %{text:.2f} Cr',
    textposition='outside',
    cliponaxis=False,
    marker=dict(color='#007A33'),
          hovertemplate='<extra></extra>'   # ⭐ GREEN BARS
)

fig.update_layout(

    paper_bgcolor='#F5F5DC',
    plot_bgcolor="#EAF1F1",

    title=dict(
        x=0.5,
        font=dict(size=22, family="Arial Black")  
    ),

    xaxis=dict(
        title='Revenue (₹ Crores)',
        ticksuffix=' Cr', 
        tickfont=dict(size=14, family="Arial Black")  
    ),

    yaxis=dict(
        title=dict(
            text='Watt',
            font=dict(size=18, family="Arial Black")   
        ),
        tickfont=dict(size=14, family="Arial Black")
    ),
     height=900,   # 🔥 increase height
    width=1600,
    margin=dict(l=120, r=80, t=80, b=50)
)
fig.show()

MONTHLY SALES ANALYSIS

Month ON MONTH TREND ANALYSIS

In [23]:
df = df_merged.copy()

In [24]:
df['PostingDate'] = pd.to_datetime(df['PostingDate'], errors='coerce')
df = df.dropna(subset=['PostingDate'])

In [25]:

df['Month'] = df['PostingDate'].dt.to_period('M').astype(str)

Step 2: Monthly Revenue calculate

In [26]:
monthly = (
    df.groupby('Month', as_index=False)['Amount']
      .sum()
      .sort_values('Month')
)

monthly['Revenue_Cr'] = monthly['Amount'] / 1e7

In [27]:
df.groupby("Month")["Amount"].sum().reset_index()

,Month,Amount
0,1-01,0.000000e+00
1,2025-02,1.197269e+08
2,2025-04,6.122005e+08
3,2025-05,7.158190e+08
4,2025-06,7.077964e+08
5,2025-07,7.479532e+08
6,2025-08,6.821527e+08
7,2025-09,7.457339e+08
8,2025-10,6.694636e+08
9,2025-11,5.450071e+08


In [28]:
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

monthly["Month"] = pd.Categorical(monthly["Month"], categories=month_order, ordered=True)
monthly = monthly.sort_values("Month")

C:\Users\naina\AppData\Local\Temp\ipykernel_33484\628112692.py:3: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  monthly["Month"] = pd.Categorical(monthly["Month"], categories=month_order, ordered=True)


In [35]:
df['PostingDate'] = pd.to_datetime(
    df['PostingDate'],
    errors='coerce'
)

In [36]:
print(df['PostingDate'].dtype)

datetime64[us]


In [37]:
print(df['PostingDate'].dt.year.unique())

[2026 2025    1]


In [38]:
print(df['PostingDate'].head(10))

0   2026-05-06
1   2025-11-06
2   2025-11-06
3   2026-03-10
4   2026-03-10
5   2026-03-10
6   2025-02-01
7   2025-02-01
8   2025-02-01
9   2025-02-01
Name: PostingDate, dtype: datetime64[us]


In [39]:
df['PostingDate'] = pd.to_datetime(df['PostingDate'])
df['Year'] = df['PostingDate'].dt.year

print(sorted(df['Year'].unique()))

[np.int32(1), np.int32(2025), np.int32(2026)]


YEAR WISE REVENUE ANALYSIS

In [ ]:
import pandas as pd
import plotly.graph_objects as go

df = df_merged.copy()

# ---------------- CLEAN DATA ----------------
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df['PostingDate'] = pd.to_datetime(df['PostingDate'], errors='coerce')
df = df.dropna(subset=['Amount', 'PostingDate'])

df['Year'] = df['PostingDate'].dt.year
df['Month'] = df['PostingDate'].dt.strftime('%b')

df = df[df['Year'] >= 2000]

month_order = [
    "Jan","Feb","Mar","Apr","May","Jun",
    "Jul","Aug","Sep","Oct","Nov","Dec"
]

years = sorted(df['Year'].unique())

# ---------------- ALL YEARS ----------------
all_monthly = (
    df.groupby('Month')['Amount']
      .sum()
      .reindex(month_order, fill_value=0)
      .reset_index()
)

all_monthly.columns = ['Month', 'Amount']
all_monthly['Revenue_Cr'] = all_monthly['Amount'] / 1e7

# ---------------- FIGURE ----------------
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=all_monthly['Month'],
        y=all_monthly['Revenue_Cr'],
        text=[f"₹ {x:.2f} Cr" for x in all_monthly['Revenue_Cr']],
        textposition='outside',
        marker_color='#007A33',
        width=0.5,
         textfont=dict(
        size=18,
        color="black"
    ),
    )
)

# ---------------- DROPDOWN BUTTONS ----------------
buttons = []

# ALL YEARS BUTTON
buttons.append(
    dict(
        label="All Years",
        method="update",
        args=[{
            "x": [all_monthly['Month']],
            "y": [all_monthly['Revenue_Cr']]
        }]
    )
)

# YEAR WISE BUTTONS (2025, 2026 etc.)
for yr in years:

    temp = df[df['Year'] == yr]

    monthly = (
        temp.groupby('Month')['Amount']
            .sum()
            .reindex(month_order, fill_value=0)
            .reset_index()
    )

    monthly.columns = ['Month', 'Amount']
    monthly['Revenue_Cr'] = monthly['Amount'] / 1e7

    buttons.append(
        dict(
            label=str(yr),
            method="update",
            args=[{
                "x": [monthly['Month']],
                "y": [monthly['Revenue_Cr']],
                "text": [[f"₹ {x:.2f} Cr" for x in monthly['Revenue_Cr']]]
            }]
        )
    )

# ---------------- LAYOUT ----------------
fig.update_layout(

    title='📊 Year wise Revenue Analysis',
    title_x=0.5,

    hoverlabel=dict(
        bgcolor="#F5F5DC",
        font=dict(color="black", size=14)
    ),

    updatemenus=[
        dict(
            buttons=buttons,
            direction='down',
            x=1.05,
            y=1.15
        )
    ],

    paper_bgcolor='#F5F5DC',
    plot_bgcolor='#EAF1F1',

    xaxis_title='Month',
    yaxis_title='Revenue (₹ Crores)',

    height=600,
    width=1200
)

fig.show()

: 